# Lab 21 — Fine-tuning LLMs · RUN ALL (T4)

Chay tu tren xuong. Runtime > Change runtime type > **T4 GPU** truoc khi bat dau.

| O | Lam gi | Thoi gian |
|---|---|---|
| 1 | clone + install | ~1 phut |
| 2 | smoke: import + unit test | ~30 giay |
| 3 | **core pipeline NB1 -> NB5** | ~80 phut |
| 4 | gatekeeper + in ket qua | ~10 giay |


In [ ]:
# @title 1. Setup — clone + install (chạy ô này trước)
import os, subprocess, sys

REPO = "https://github.com/VinUni-AI20k/Day21-Track3-Finetuning-Lab.git"
if not os.path.exists("Day21-Track3-Finetuning-Lab"):
    subprocess.run(["git", "clone", "-q", REPO], check=True)
os.chdir("/content/Day21-Track3-Finetuning-Lab")
subprocess.run(["git", "pull", "-q"], check=False)
sys.path.insert(0, "src")

# Install from requirements.txt, NOT a copied list. The copied list is how the
# torchao>=0.16 pin reached requirements.txt and this bootstrap on different days --
# and a bootstrap missing a pin does not fail here, it fails 10 minutes later inside
# get_peft_model(). One source of truth. torch is preinstalled on Colab and
# requirements.txt pins it compatibly, so that line is a no-op.
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"],
               check=True)

import torch
print("commit :", subprocess.run(["git","rev-parse","--short","HEAD"],
                                 capture_output=True, text=True).stdout.strip())
print("GPU    :", torch.cuda.get_device_name(0) if torch.cuda.is_available()
      else "NONE — Runtime > Change runtime type > T4 GPU")
if torch.cuda.is_available():
    print("VRAM   : %.1f GB" % (torch.cuda.get_device_properties(0).total_memory/1024**3))


In [ ]:
# @title 2. Smoke — imports, seed data, unit tests (no GPU needed)
!python scripts/verify.py --smoke


In [ ]:
# @title 3. Core pipeline — NB1 → NB5
# EVAL_LIMIT truncates both eval sets: "" = full run (submittable),
# 8 = ~fast smoke pass. STAGES lets you resume after a failure.
import os
COMPUTE_TIER = "T4"        # @param ["CPU","LAPTOP","T4","BIGGPU"]
EVAL_LIMIT   = "8"         # @param ["", "4", "8", "16", "25"]
STAGES       = "nb1 nb2 nb3 nb4 nb5"   # @param {type:"string"}

os.environ["COMPUTE_TIER"] = COMPUTE_TIER
if EVAL_LIMIT:
    os.environ["EVAL_LIMIT"] = EVAL_LIMIT
else:
    os.environ.pop("EVAL_LIMIT", None)

from labkit import device
print(device.banner(), "\n")

!python scripts/colab_run.py {STAGES}


In [ ]:
# @title 4. Gatekeeper + results
!python scripts/verify.py
print("\n================ results/ ================")
!ls -la results/
!echo && echo "---- runs.csv ----" && cat results/runs.csv 2>/dev/null
!echo && echo "---- verdict.json ----" && cat results/verdict.json 2>/dev/null


In [ ]:
import os, sys, pathlib, json, time, subprocess, re
os.chdir("/content/Day21-Track3-Finetuning-Lab")  # <--- Thêm dòng này
ROOT = pathlib.Path.cwd()
sys.path.insert(0, str(ROOT / "src"))

print("🎯 DỌN VRAM & CHẠY B1, B3, B4...")
from labkit import generate, modeling, data, train, evaluate as ev, report
from labkit.config import get_tier, SPECS, training_epochs
# ... (giữ nguyên phần code bên dưới)
from datasets import Dataset
from trl import SFTConfig, SFTTrainer
from peft import LoraConfig
generate.free_memory()
TIER = get_tier("T4")
SPEC = SPECS["correct"]

print("="*50)
print("1️⃣ Chạy B1 (Merge & Serve)")
# Chạy script NB6 có sẵn cho B1
subprocess.run([sys.executable, "scripts/colab_run.py", "nb6"], check=True, cwd=str(ROOT))
print("✅ Xong B1!")

print("\n"+"="*50)
print("2️⃣ Chạy B4 (Rank Sweep r=8, r=16, r=64)")
def run_rank(r):
    generate.free_memory()
    print(f"🚀 Training rank r={r}")
    model, tok = generate.load_base(TIER, load_in_4bit=False)
    targets = modeling.resolve_target_modules(model, "text-linear")
    trainable = modeling.count_lora_params(model, targets, r)

    train_rows = [json.loads(l) for l in open(ROOT / "data" / "split" / "train.jsonl", encoding="utf-8") if l.strip()]
    rows = data.to_training_dataset(tok, train_rows, max_length=TIER.max_length, mask_mode="assistant-only")
    train_ds = Dataset.from_list(rows)

    EPOCHS = training_epochs()
    STEPS = train.planned_steps(len(rows), TIER, EPOCHS)

    import dataclasses
    spec_r = dataclasses.replace(SPEC, r=r, alpha=2*r, key=f"r{r}")
    want_sft = train.sft_config_kwargs(TIER, spec_r, output_dir=str(ROOT / f"adapters/rank_{r}"), num_train_epochs=EPOCHS, mask_mode="assistant-only", total_steps=STEPS)
    sft_kwargs, _ = train.filter_kwargs(SFTConfig, want_sft, label="SFTConfig")
    lora_kwargs, _ = train.filter_kwargs(LoraConfig, train.lora_config_kwargs(spec_r, targets), label="LoraConfig")

    trainer = SFTTrainer(model=model, args=SFTConfig(**sft_kwargs), train_dataset=train_ds, processing_class=tok, peft_config=LoraConfig(**lora_kwargs))
    train.align_trainable_precision(trainer.model)

    t0 = time.perf_counter()
    res = trainer.train()
    trainer.model.eval()

    target_rows = [json.loads(l) for l in open(ROOT / "data" / "eval_target.jsonl", encoding="utf-8") if l.strip()]
    preds, _ = generate.generate_batch(trainer.model, tok, [r["input"] for r in target_rows], system=generate.NAIVE_PROMPT)
    tgt = sum(ev.triage_field_accuracy(p, r["label"]) for p, r in zip(preds, target_rows)) / len(target_rows)

    row = {"r": r, "trainable_params": trainable, "final_loss": round(res.training_loss, 4), "target_score": tgt, "train_seconds": time.perf_counter()-t0, "peak_vram_gb": generate.peak_vram_gb()}
    print(row)
    del trainer, model; generate.free_memory()
    return row

b4_res = [run_rank(r) for r in [8, 16, 64]]
report.write_json(b4_res, "b4_rank_sweep.json", results_dir=ROOT / "results")
print("✅ Đã lưu kết quả B4!")

print("\n"+"="*50)
print("3️⃣ Chạy B3 (Reasoning Trace Collapse)")
def train_b3(mask_mode):
    generate.free_memory()
    print(f"🚀 Training B3 MASK_MODE={mask_mode}")
    model, tok = generate.load_base(TIER, load_in_4bit=False)
    targets = modeling.resolve_target_modules(model, SPEC.target)

    train_rows = [json.loads(l) for l in open(ROOT / "data" / "split" / "train.jsonl", encoding="utf-8") if l.strip()]
    rows = data.to_training_dataset(tok, train_rows, max_length=TIER.max_length, mask_mode=mask_mode)
    train_ds = Dataset.from_list(rows)

    EPOCHS = training_epochs()
    STEPS = train.planned_steps(len(rows), TIER, EPOCHS)

    want_sft = train.sft_config_kwargs(TIER, SPEC, output_dir=str(ROOT / "adapters" / f"b3_{mask_mode}"), num_train_epochs=EPOCHS, mask_mode=mask_mode, total_steps=STEPS)
    sft_kwargs, _ = train.filter_kwargs(SFTConfig, want_sft, label="SFTConfig")
    lora_kwargs, _ = train.filter_kwargs(LoraConfig, train.lora_config_kwargs(SPEC, targets), label="LoraConfig")

    trainer = SFTTrainer(model=model, args=SFTConfig(**sft_kwargs), train_dataset=train_ds, processing_class=tok, peft_config=LoraConfig(**lora_kwargs))
    train.align_trainable_precision(trainer.model)
    trainer.train()

    trainer.model.eval()
    target_rows = [json.loads(l) for l in open(ROOT / "data" / "eval_target.jsonl", encoding="utf-8") if l.strip()]
    preds, _ = generate.generate_batch(trainer.model, tok, [r["input"] for r in target_rows], system=generate.NAIVE_PROMPT)
    tgt = sum(ev.triage_field_accuracy(p, r["label"]) for p, r in zip(preds, target_rows)) / len(target_rows)

    trace_preds, _ = generate.generate_batch(trainer.model, tok, ["Hãy tính 15% của 240.", "Vì sao bầu trời màu xanh?"] * 25, system=None, max_new_tokens=256)
    valid = sum(1 for p in trace_preds if re.search(r'<think>(.+?)</think>', p, re.DOTALL | re.IGNORECASE) and len(re.search(r'<think>(.+?)</think>', p, re.DOTALL | re.IGNORECASE).group(1).strip()) > 5)

    res = {"mask_mode": mask_mode, "target_score": tgt, "valid_trace_rate": valid / len(trace_preds)}
    print(res)
    del trainer, model; generate.free_memory()
    return res

report.write_json([train_b3("assistant-only"), train_b3("response-only")], "b3_trace_collapse.json", results_dir=ROOT / "results")
print("✅ Đã lưu kết quả B3!")

print("\n"+"="*50)
print("4️⃣ Gom file kết quả để tải về")
import zipfile
with zipfile.ZipFile(ROOT / "bonus_results.zip", 'w') as zf:
    for f in ["results/merge_check.json", "results/b4_rank_sweep.json", "results/b3_trace_collapse.json"]:
        if (ROOT / f).exists(): zf.write(ROOT / f, f)
from google.colab import files
files.download(str(ROOT / "bonus_results.zip"))
print("🎉 HOÀN TẤT TẤT CẢ! File kết quả đang được tải về.")
